In [ ]:
from strats.oscillator_multi_timestamp_strategy import OscillatorMultiTimeStampStrategy
from models.backtesting_models import AccountState
from backtesting_engine import BacktestEngine
from risk_manager import RiskManager
from utils.misc import resample_ohlcv
from dataclasses import asdict
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import copy

import pandas as pd
from models.backtesting_models import TradeSignal

In [ ]:
# Load pre-fetched 1m data from parquet
market_bars_df = pd.read_parquet("../../data/hyperliquid_1m_all_symbols.parquet")
funding_df = pd.read_parquet("../../data/hyperliquid_funding_rates.parquet")

# Clean timestamp
funding_df["timestamp"] = funding_df["timestamp"].dt.floor("h")


symbol_data = {}
for symbol in market_bars_df['symbol'].unique():
    if symbol == "HYPE":
        continue # Skip HYPE
    df_1m_temp = market_bars_df[market_bars_df['symbol'] == symbol].reset_index(drop=True)
    df_funding_temp = funding_df[funding_df['symbol'] == symbol].reset_index(drop=True)
    
    df_1m_temp["timestamp"] = pd.to_datetime(df_1m_temp["timestamp"], utc=True)
    df_funding_temp["timestamp"] = pd.to_datetime(df_funding_temp["timestamp"], utc=True)
    
    
    symbol_data[symbol] = {
        "1m": df_1m_temp, 
        "5m": resample_ohlcv(df_1m_temp, "5min"),
        "15m": resample_ohlcv(df_1m_temp, "15min"), 
        "funding": df_funding_temp.drop_duplicates(subset=["timestamp"], keep="last").reset_index(drop=True)
    }

In [ ]:
strategy = OscillatorMultiTimeStampStrategy()
account = AccountState(balance=10_000.0)
engine = BacktestEngine(allow_same_bar_exit=False)
risk_manager = RiskManager(
    leverage=10,
    min_notional=100.0,
    max_absolute_margin=300.0,
)
start_time = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=1)

all_signals = []
for symbol, data_dict in symbol_data.items():
    df_1m = data_dict['1m']
    df_funding = data_dict['funding'].astype({"timestamp": "datetime64[ns, UTC]", "funding_rate_relative": "float64"})
    df_15m_stochastic = strategy._build_stochastic(data_dict['15m'])
    df_5m_stochastic = strategy._build_stochastic(data_dict['5m'])

    signals = strategy.generate_trade_signals(
        symbol=symbol, 
        df_1m=df_1m, 
        df_5m=df_5m_stochastic,
        df_15m=df_15m_stochastic, 
        funding_df=df_funding,
        start_time=start_time
    )

    print(f"{symbol}: Generated {len(signals)} signals!")
    all_signals.extend(signals)

all_signals_df = pd.DataFrame([asdict(sig) for sig in all_signals])
all_signals_df = all_signals_df.sort_values(["timestamp", "symbol"]).reset_index(drop=True)
signals_by_timestamp: dict[pd.Timestamp, list[TradeSignal]] = defaultdict(list)
for sig in all_signals:
    ts = pd.to_datetime(sig.timestamp, utc=True)
    signals_by_timestamp[ts].append(sig)

bars_1m_by_symbol = {sym: bars["1m"].set_index('timestamp') for sym, bars in symbol_data.items() if "1m" in bars}
funding_by_symbol = {sym: bars["funding"].set_index('timestamp')for sym, bars in symbol_data.items() if "funding" in bars}
all_timestamps = market_bars_df['timestamp'].drop_duplicates().sort_values()    


In [ ]:
for sig in all_signals:
    data_dict = symbol_data[sig.symbol]
    df_1m = data_dict['1m']
    df_funding = data_dict['funding']
    df_15m_stochastic = strategy._build_stochastic(data_dict['15m'])
    df_5m_stochastic = strategy._build_stochastic(data_dict['5m'])
    fig = strategy.plot_signal(
                sig, 
                df_1m=df_1m, 
                df_5m= df_5m_stochastic,
                df_15m=df_15m_stochastic, 
                funding_df=df_funding,
                lookahead_mins=60, 
                lookback_mins=60,
                return_fig=False,)

In [ ]:
sig

In [ ]:
sig

## Backtesting Loop

In [ ]:
equity_history = []
all_rejected_signals = []
pending_execution_signals = {symbol: None for symbol in symbol_data.keys()}


# Main simulation loop
for ts in all_timestamps:
    print(f"\n=== Processing timestamp: {ts} ===")
    
    if (0 <= ts.hour < 4) or (12 <= ts.hour < 16):
        new_signals = signals_by_timestamp.get(ts, [])
    else:
        new_signals = []
    
    execution_batch = {}
    
    # 1. Get current market snapshot
    current_bars = {symbol: df.loc[ts] for symbol, df in bars_1m_by_symbol.items()}
    current_prices = {symbol: bar["close"] for symbol, bar in current_bars.items()}
    current_funding = {
        symbol: 
        funding_by_symbol[symbol].loc[ts]["funding_rate_relative"] if ts in funding_by_symbol[symbol].index else 0.0 
        for symbol in symbol_data.keys()
        }
    
    
    account = engine.apply_funding_rate(account, funding_rates=current_funding, current_prices=current_prices)
    # 2. See if any existing trades closed on this bar
    account = engine.update_account(account, current_bars)   
    
    # 3. CALCULATE equity and free margin    
    # current_equity = account.get_equity(current_prices)
    current_free_margin = account.get_free_margin(account.balance)
    
    # 4. Risk Manager: Size new signals based on current equity and free margin, and build execution batch
    if new_signals:
        execution_batch, rejected = risk_manager.build_execution_batch(
            ts=ts,
            signals=new_signals,
            df_1m=bars_1m_by_symbol,
            free_margin=current_free_margin
        )
        # Track rejections across the whole backtest
        all_rejected_signals.extend(rejected)
    
    # 5. Execution
    execution_signals_sorted = sorted(execution_batch, key=lambda item: item.conviction if item else 0, reverse=True)
    for exec_sig in execution_signals_sorted:
        symbol = exec_sig.symbol
        bar = current_bars.get(symbol)
        assert bar is not None, f"Bar data missing for symbol {symbol} at timestamp {ts}"
        
        new_pos = engine.try_fill_execution(account, exec_sig, bar)
        if new_pos is not None:
            account.open_positions.append(new_pos)
                
    # 6. SNAPSHOT: Record equity and signals
    current_equity = account.get_equity(current_prices)
    equity_history.append({
        "timestamp": ts,
        "equity": current_equity, 
        "balance": account.balance,
        "open_count": len(account.open_positions),
        "rejected_count": len(all_rejected_signals),
    })


## Results

In [ ]:
final_balance = account.balance
total_trades = account.closed_trades
equity_df = pd.DataFrame(equity_history) 
trades_df = pd.DataFrame([asdict(t) | t.metadata for t in total_trades])
trades_df['entry_delay'] = (trades_df['entry_time'] - trades_df['crossover_timestamp']).dt.total_seconds() / 60
# trades_df['k_gap'] = abs(trades_df['entry_slow_k'] - trades_df['crossing_slow_k'])

wins_net = [t for t in total_trades if t.pnl_net > 0]
wins_gross = [t for t in total_trades if t.pnl_gross > 0]
win_rate = (len(wins_net) / len(total_trades) * 100) if total_trades else 0
profit_factor = trades_df[trades_df['pnl_net'] > 0]['pnl_net'].sum() / abs(trades_df[trades_df['pnl_net'] < 0]['pnl_net'].sum())
positive_trades = trades_df[trades_df['pnl_net'] > 0]
trade_profit = positive_trades['pnl_net'].sum()
negative_trades = trades_df[trades_df['pnl_net'] < 0]
trade_losses = negative_trades['pnl_net'].sum()
avg_positive_trade_profit = positive_trades['pnl_net'].mean()
avg_negative_trade_loss = negative_trades['pnl_net'].mean()
total_funding = trades_df['cumulative_funding_payment'].sum()
total_fees = -(trades_df['pnl_gross'] - trades_df['pnl_net']).sum()
number_of_days = (equity_df['timestamp'].max() - equity_df['timestamp'].min()).days

In [ ]:
print(f"Number of days:         {number_of_days}")
print(f"Profit over {number_of_days} days: ${final_balance - 10_000:,.2f}")
print(f"Per Day Profit:           ${(final_balance - 10_000) / number_of_days:,.2f}")
print(f"Final balance:          ${final_balance:,.2f}")
print(f"Total Trades:           {len(total_trades):,}")
print("Trades per Day:          {:.2f}".format(len(total_trades) / number_of_days))
print(f"Win rate (net):         {(len(wins_net) / len(total_trades) * 100) if total_trades else 0:.1f}%")
print(f"Win rate (gross):       {(len(wins_gross) / len(total_trades) * 100) if total_trades else 0:.1f}%")
print(f"Open positions:         {len(account.open_positions):,}")
print(f"Profits/Losses:         ${trade_profit:,.2f} / ${trade_losses:,.2f}")
print(f"Funding Total:          ${total_funding:,.2f}")
print(f"Fees Total:             ${total_fees:,.2f}")
print(f"Profit Factor:          {profit_factor:.2f}")
print(f"Average PnL per positive trade: ${avg_positive_trade_profit:,.2f}")
print(f"Average PnL per negative trade: ${avg_negative_trade_loss:,.2f}")
print(f"Number of rejected signals: {len(all_rejected_signals):,}")

Understand the trades

In [ ]:
# Investigate trade duration
positive_trades['duration'] = (positive_trades['exit_time'] - positive_trades['entry_time']).dt.total_seconds() / 60 / 60
negative_trades['duration'] = (negative_trades['exit_time'] - negative_trades['entry_time']).dt.total_seconds() / 60 / 60

print(f"Average duration of positive trades: {positive_trades['duration'].mean():.2f} hours")
print(f"Average duration of negative trades: {negative_trades['duration'].mean():.2f} hours")

In [ ]:
negative_trades['k_gap'].hist(bins=50)


In [ ]:

from scipy import stats as scipy_stats

df = trades_df.copy()
df["entry_time"] = pd.to_datetime(df["entry_time"], utc=True, errors="coerce")
df["crossover_timestamp"] = pd.to_datetime(df["crossover_timestamp"], utc=True, errors="coerce")
df["is_win"] = df["pnl_net"] > 0
df["duration_h"] = (pd.to_datetime(df["exit_time"], utc=True) - df["entry_time"]).dt.total_seconds() / 3600
df["ema_distance_pct"] = (df["entry"] - df["ema_close"]) / df["ema_close"] * 100
df["funding_aligned"] = (
((df["direction"] == "LONG") & (df["last_funding_rate"] < 0)) |
((df["direction"] == "SHORT") & (df["last_funding_rate"] > 0))
)

wins = df[df["is_win"]]
losses = df[~df["is_win"]]


rows = []
for label, col in [
("Crossing Slow K", "crossing_slow_k"),
("Entry Slow K", "entry_slow_k"),
("K Gap (entry−cross)", "k_gap"),
("Entry Delay (min)", "entry_delay"),
("Last Funding Rate", "last_funding_rate"),
("EMA Distance %", "ema_distance_pct"),
("Duration (h)", "duration_h"),
]:
    w, l = wins[col].dropna(), losses[col].dropna()
    _, p = scipy_stats.ttest_ind(w, l, equal_var=False)
    rows.append({
    "Feature": label,
    "Winners (mean)": f"{w.mean():.3f}",
    "Losers (mean)": f"{l.mean():.3f}",
    "Diff": f"{w.mean() - l.mean():+.3f}",
    "p-value": f"{p:.3f}",
    "Significant": "✅" if p < 0.05 else "—",
    })
print("── Numeric Feature Comparison (Welch t-test) ──────────────────────────")
print(pd.DataFrame(rows).to_string(index=False))


print("\n── Win Rate by Direction ───────────────────────────────────────────────")
dir_stats = df.groupby("direction").agg(
trades=("is_win", "size"), win_rate=("is_win", "mean"),
avg_pnl=("pnl_net", "mean"), total_pnl=("pnl_net", "sum"),
)
dir_stats["win_rate"] = dir_stats["win_rate"].map("{:.1%}".format)
dir_stats["avg_pnl"] = dir_stats["avg_pnl"].map("${:.2f}".format)
dir_stats["total_pnl"] = dir_stats["total_pnl"].map("${:.2f}".format)
print(dir_stats.to_string())


print("\n── Exit Reason Distribution ────────────────────────────────────────────")
exit_stats = df.groupby(["exit_reason", "is_win"]).agg(
count=("pnl_net", "size"), avg_pnl=("pnl_net", "mean")
).rename(index={True: "Win", False: "Loss"}, level="is_win")
print(exit_stats.to_string())

print("\n── Win Rate: Funding Aligned vs Against ────────────────────────────────")
fund_stats = df.groupby("funding_aligned").agg(
trades=("is_win", "size"), win_rate=("is_win", "mean"), avg_pnl=("pnl_net", "mean")
)
fund_stats.index = fund_stats.index.map({True: "Aligned", False: "Against"})
fund_stats["win_rate"] = fund_stats["win_rate"].map("{:.1%}".format)
fund_stats["avg_pnl"] = fund_stats["avg_pnl"].map("${:.2f}".format)
print(fund_stats.to_string())


print("\n── LONG trades: EMA side ───────────────────────────────────────────────")
longs = df[df["direction"] == "LONG"].copy()
longs["ema_side"] = longs["ema_distance_pct"].apply(lambda x: "Below EMA" if x < 0 else "Above EMA")
ema_stats = longs.groupby("ema_side").agg(
trades=("is_win", "size"), win_rate=("is_win", "mean"), avg_pnl=("pnl_net", "mean")
)
ema_stats["win_rate"] = ema_stats["win_rate"].map("{:.1%}".format)
ema_stats["avg_pnl"] = ema_stats["avg_pnl"].map("${:.2f}".format)
print(ema_stats.to_string())


plot_cols = ["crossing_slow_k", "entry_slow_k", "k_gap", "entry_delay", "last_funding_rate", "ema_distance_pct"]
plot_labels = ["Crossing Slow K", "Entry Slow K", "K Gap", "Entry Delay (min)", "Last Funding Rate", "EMA Distance %"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col, label in zip(axes.flatten(), plot_cols, plot_labels):
    w_vals, l_vals = wins[col].dropna(), losses[col].dropna()
    ax.hist(l_vals, bins=30, alpha=0.55, color="#ff4d6d", label="Loss", density=True)
    ax.hist(w_vals, bins=30, alpha=0.55, color="#00c896", label="Win", density=True)
    ax.axvline(w_vals.mean(), color="#00c896", lw=1.5, ls="--")
    ax.axvline(l_vals.mean(), color="#ff4d6d", lw=1.5, ls="--")
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Signal Feature Distributions: Winners vs Losers", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
!pip install scipy

In [ ]:
def duration_bucket(h):
    if h <= 12:
        return f"{h}h"
    elif h <= 24:
        return "12-24h"
    elif h <= 48:
        return "1-2d"
    elif h <= 72:
        return "2-3d"
    else:
        return f"{h // 24}d+"

bucket_order = [f"{h}h" for h in range(0, 13)] + ["12-24h", "1-2d", "2-3d"]

frame = positive_trades.copy()
frame['duration_h'] = (frame['exit_time'] - frame['entry_time']).dt.total_seconds() / 3600
frame['duration_h'] = frame['duration_h'].astype(int)
frame['duration_bucket'] = frame['duration_h'].apply(duration_bucket)
# Add any extra buckets that appear beyond 3d
extra = [b for b in frame['duration_bucket'].unique() if b not in bucket_order]
all_buckets = bucket_order + sorted(extra)
frame['duration_bucket'] = pd.Categorical(frame['duration_bucket'], categories=all_buckets, ordered=True)
frame["is_win"] = frame["pnl_net"] > 0

stats = frame.groupby(["symbol", "duration_bucket"], observed=False).agg(
    sum_pnl=("pnl_net", "sum"),
    trade_count=("pnl_net", "size"),
    hit_rate=("is_win", "mean"),
)

sum_pnl_map  = stats["sum_pnl"].unstack("duration_bucket")
count_map    = stats["trade_count"].unstack("duration_bucket")
hit_rate_map = (stats["hit_rate"] * 100).unstack("duration_bucket")

agg = frame.groupby("duration_bucket", observed=False).agg(sum_pnl=("pnl_net", "sum"), trade_count=("pnl_net", "size"), hit_rate=("is_win", "mean"))
sum_pnl_map.loc["ALL"]  = agg["sum_pnl"]
count_map.loc["ALL"]    = agg["trade_count"]
hit_rate_map.loc["ALL"] = agg["hit_rate"] * 100

h = max(6, len(sum_pnl_map) * 0.4)

plt.figure(figsize=(18, h))
sns.heatmap(sum_pnl_map, annot=True, fmt=".0f", cmap="RdYlGn", center=0, linewidths=0.4, linecolor="white", cbar_kws={"label": "Total PnL ($)"})
plt.title("Total PnL by Symbol and Trade Duration")
plt.xlabel("Duration")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, h))
sns.heatmap(count_map, annot=True, fmt=".0f", cmap="Blues", linewidths=0.4, linecolor="white", cbar_kws={"label": "Trades"})
plt.title("Trade Count by Symbol and Trade Duration")
plt.xlabel("Duration")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, h))
sns.heatmap(hit_rate_map, annot=True, fmt=".0f", cmap="YlGn", vmin=0, vmax=100, linewidths=0.4, linecolor="white", cbar_kws={"label": "Hit Rate (%)"})
plt.title("Hit Rate by Symbol and Trade Duration")
plt.xlabel("Duration")
plt.tight_layout()
plt.show()


In [ ]:
hour_df = trades_df.copy()
hour_df["entry_time"] = pd.to_datetime(hour_df["entry_time"], utc=True, errors="coerce")
hour_df = hour_df.dropna(subset=["entry_time", "pnl_net"])
hour_df["is_win"] = hour_df["pnl_net"] > 0
hour_df["entry_hour"] = hour_df["entry_time"].dt.hour

hour_stats = (
    hour_df.groupby(["symbol", "entry_hour"])
    .agg(
        trades=("pnl_net", "size"),
        sum_pnl=("pnl_net", "sum"),
        win_rate=("is_win", "mean"),
    )
)

all_hours = list(range(24))
trades_map   = hour_stats["trades"].unstack("entry_hour").reindex(columns=all_hours)
sum_pnl_map  = hour_stats["sum_pnl"].unstack("entry_hour").reindex(columns=all_hours)
win_rate_map = hour_stats["win_rate"].unstack("entry_hour").reindex(columns=all_hours)

# ALL row
agg = hour_df.groupby("entry_hour").agg(trades=("pnl_net", "size"), sum_pnl=("pnl_net", "sum"), win_rate=("is_win", "mean"))
trades_map.loc["ALL"]   = agg["trades"].reindex(all_hours)
sum_pnl_map.loc["ALL"]  = agg["sum_pnl"].reindex(all_hours)
win_rate_map.loc["ALL"] = agg["win_rate"].reindex(all_hours)

h = max(6, len(trades_map) * 0.35)

plt.figure(figsize=(20, h))
sns.heatmap(sum_pnl_map, annot=True, fmt=".0f", cmap="RdYlGn", center=0, linewidths=0.2, linecolor="white", cbar_kws={"label": "Total PnL ($)"})
plt.title("Total PnL by Symbol and Hour of Day (UTC)")
plt.xlabel("UTC Hour")
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, h))
sns.heatmap(trades_map, annot=True, fmt=".0f", cmap="Blues", linewidths=0.2, linecolor="white", cbar_kws={"label": "Trades"})
plt.title("Trade Count by Symbol and Hour of Day (UTC)")
plt.xlabel("UTC Hour")
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, h))
sns.heatmap(win_rate_map * 100, annot=True, fmt=".0f", cmap="YlGn", vmin=0, vmax=100, linewidths=0.2, linecolor="white", cbar_kws={"label": "Win Rate (%)"})
plt.title("Win Rate by Symbol and Hour of Day (UTC)")
plt.xlabel("UTC Hour")
plt.tight_layout()
plt.show()


In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

frame = trades_df.copy()
frame["entry_time"] = pd.to_datetime(frame["entry_time"], utc=True, errors="coerce")
frame = frame.dropna(subset=["entry_time", "pnl_net"])
frame["entry_weekday"] = pd.Categorical(
    frame["entry_time"].dt.day_name(), categories=weekday_order, ordered=True
)
frame["entry_hour"] = frame["entry_time"].dt.hour
frame["is_win"] = frame["pnl_net"] > 0

stats = frame.groupby(["entry_weekday", "entry_hour"], observed=False).agg(
    sum_pnl=("pnl_net", "sum"),
    trade_count=("pnl_net", "size"),
    hit_rate=("is_win", "mean"),
)

sum_pnl_map = stats["sum_pnl"].unstack("entry_hour").reindex(weekday_order)
count_map = stats["trade_count"].unstack("entry_hour").reindex(weekday_order)
hit_rate_map = (stats["hit_rate"] * 100).unstack("entry_hour").reindex(weekday_order)

plt.figure(figsize=(14, 5))
sns.heatmap(
    sum_pnl_map,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Total PnL ($)"},
)
plt.title("Total PnL")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()

plt.figure(figsize=(14, 5))
sns.heatmap(
    count_map,
    cmap="Blues",
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Trades"},
)
plt.title("Number of Trades")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()

plt.figure(figsize=(14, 5))
sns.heatmap(
    hit_rate_map,
    cmap="YlGn",
    vmin=0,
    vmax=100,
    annot=True,
    fmt=".0f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Positive Hit Rate (%)"},
)
plt.title("Positive Trade Hit Rate")
plt.xlabel("UTC hour")
plt.ylabel("")
plt.show()

In [ ]:
minute_labels = [f"{i*5}-{(i+1)*5}" for i in range(12)]

minute_df = trades_df.copy()
minute_df["entry_time"] = pd.to_datetime(minute_df["entry_time"], utc=True, errors="coerce")
minute_df = minute_df.dropna(subset=["entry_time", "pnl_net"])
minute_df["is_win"] = minute_df["pnl_net"] > 0
minute_df["minute_bucket"] = pd.cut(
    minute_df["entry_time"].dt.minute,
    bins=list(range(0, 65, 5)),
    labels=minute_labels,
    include_lowest=True,
    right=False,
)

minute_stats = (
    minute_df.groupby(["symbol", "minute_bucket"], observed=False)
    .agg(
        trades=("pnl_net", "size"),
        avg_pnl=("pnl_net", "mean"),
        win_rate=("is_win", "mean"),
    )
)

trades_map   = minute_stats["trades"].unstack("minute_bucket").reindex(columns=minute_labels)
avg_pnl_map  = minute_stats["avg_pnl"].unstack("minute_bucket").reindex(columns=minute_labels)
win_rate_map = minute_stats["win_rate"].unstack("minute_bucket").reindex(columns=minute_labels)

plt.figure(figsize=(18, max(6, len(trades_map) * 0.22)))
sns.heatmap(
    trades_map,
    cmap="Blues",
    annot=True,
    fmt=".0f",
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Trades"},
)
plt.title("Trade Count by Symbol and Minute-of-Hour Bucket")
plt.xlabel("Minute Bucket")
plt.ylabel("Symbol")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, max(6, len(avg_pnl_map) * 0.22)))
sns.heatmap(
    avg_pnl_map,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Average PnL ($)"},
)
plt.title("Average PnL by Symbol and Minute-of-Hour Bucket")
plt.xlabel("Minute Bucket")
plt.ylabel("Symbol")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18, max(6, len(win_rate_map) * 0.22)))
sns.heatmap(
    win_rate_map,
    cmap="YlGn",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".0%",
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Win Rate"},
)
plt.title("Win Rate by Symbol and Minute-of-Hour Bucket")
plt.xlabel("Minute Bucket")
plt.ylabel("Symbol")
plt.tight_layout()
plt.show()


In [ ]:
negative_trades[['symbol', 'direction', 'entry_time', 'entry', 'qty', 'sl', 'tp', 'liq_price', 'pnl_net', 'pnl_gross', 'exit_reason', 'exit_time', 'close_price']]

In [ ]:
liq_df = trades_df.copy()
liq_df["entry"]     = pd.to_numeric(liq_df["entry"])
liq_df["liq_price"] = pd.to_numeric(liq_df["liq_price"])
liq_df["tp"]        = pd.to_numeric(liq_df["tp"])
liq_df["sl"]        = pd.to_numeric(liq_df["sl"])
liq_df["qty"]       = pd.to_numeric(liq_df["qty"])
liq_df["margin_req"] = pd.to_numeric(liq_df["margin_req"])

# Derive effective leverage from the trade: notional / margin
liq_df["notional"]     = liq_df["entry"] * liq_df["qty"]
liq_df["eff_leverage"] = liq_df["notional"] / liq_df["margin_req"]

# Raw price % move to hit liq / sl / tp
liq_df["liq_price_pct"] = (liq_df["liq_price"] - liq_df["entry"]).abs() / liq_df["entry"] * 100
liq_df["sl_price_pct"]  = (liq_df["sl"]         - liq_df["entry"]).abs() / liq_df["entry"] * 100
liq_df["tp_price_pct"]  = (liq_df["tp"]         - liq_df["entry"]).abs() / liq_df["entry"] * 100

# Effective PnL % on margin (price % * leverage)
liq_df["liq_pnl_pct"] = liq_df["liq_price_pct"] * liq_df["eff_leverage"]
liq_df["sl_pnl_pct"]  = liq_df["sl_price_pct"]  * liq_df["eff_leverage"]
liq_df["tp_pnl_pct"]  = liq_df["tp_price_pct"]  * liq_df["eff_leverage"]

# How much of the TP move you'd need in reverse to get liquidated / stopped
liq_df["liq_vs_tp"] = liq_df["liq_price_pct"] / liq_df["tp_price_pct"]
liq_df["sl_vs_tp"]  = liq_df["sl_price_pct"]  / liq_df["tp_price_pct"]

summary = (
    liq_df.groupby("symbol")
    .agg(
        trades=("entry",              "size"),
        avg_leverage=("eff_leverage", "mean"),
        liq_price_pct=("liq_price_pct", "mean"),
        sl_price_pct=("sl_price_pct",   "mean"),
        tp_price_pct=("tp_price_pct",   "mean"),
        liq_pnl_pct=("liq_pnl_pct",    "mean"),
        sl_pnl_pct=("sl_pnl_pct",      "mean"),
        tp_pnl_pct=("tp_pnl_pct",      "mean"),
        liq_vs_tp=("liq_vs_tp",        "mean"),
        sl_vs_tp=("sl_vs_tp",          "mean"),
    )
    .sort_values("liq_price_pct")
)

summary["avg_leverage"]  = summary["avg_leverage"].map("{:.1f}x".format)
summary["liq_price_pct"] = summary["liq_price_pct"].map("{:.2f}%".format)
summary["sl_price_pct"]  = summary["sl_price_pct"].map("{:.2f}%".format)
summary["tp_price_pct"]  = summary["tp_price_pct"].map("{:.2f}%".format)
summary["liq_pnl_pct"]  = summary["liq_pnl_pct"].map("{:.1f}%".format)
summary["sl_pnl_pct"]   = summary["sl_pnl_pct"].map("{:.1f}%".format)
summary["tp_pnl_pct"]   = summary["tp_pnl_pct"].map("{:.1f}%".format)
summary["liq_vs_tp"]    = summary["liq_vs_tp"].map("{:.2f}x".format)
summary["sl_vs_tp"]     = summary["sl_vs_tp"].map("{:.2f}x".format)

summary.columns = [
    "Trades", "Avg Leverage",
    "Liq Dist (price %)", "SL Dist (price %)", "TP Dist (price %)",
    "Liq Loss (PnL %)", "SL Loss (PnL %)", "TP Gain (PnL %)",
    "Liq/TP Ratio", "SL/TP Ratio",
]
summary



In [ ]:
# MAE (Maximum Adverse Excursion) for winning trades
# = furthest price went against you before eventually hitting TP

winning_trades = trades_df[trades_df["exit_reason"] == "TP"].copy()
winning_trades["entry_time"] = pd.to_datetime(winning_trades["entry_time"], utc=True)
winning_trades["exit_time"]  = pd.to_datetime(winning_trades["exit_time"],  utc=True)
winning_trades["entry"]      = pd.to_numeric(winning_trades["entry"])

mae_rows = []
for _, trade in winning_trades.iterrows():
    bars = bars_1m_by_symbol.get(trade["symbol"])
    if bars is None:
        continue
    trade_bars = bars.loc[trade["entry_time"]:trade["exit_time"]]
    if trade_bars.empty:
        continue

    if trade["direction"] == "LONG":
        worst_price = trade_bars["low"].min()
        mae_price   = trade["entry"] - worst_price
    else:
        worst_price = trade_bars["high"].max()
        mae_price   = worst_price - trade["entry"]

    mae_pct = mae_price / trade["entry"] * 100
    mae_rows.append({"symbol": trade["symbol"], "mae_price": mae_price, "mae_pct": mae_pct})

mae_df = pd.DataFrame(mae_rows)

print(f"Winning trades analysed: {len(mae_df):,}")
print(f"Average MAE across all symbols: {mae_df['mae_pct'].mean():.2f}%")
print(f"Median  MAE across all symbols: {mae_df['mae_pct'].median():.2f}%")

summary = (
    mae_df.groupby("symbol")
    .agg(
        trades=("mae_pct", "size"),
        mae_pct_mean=("mae_pct", "mean"),
        mae_pct_median=("mae_pct", "median"),
        mae_pct_max=("mae_pct", "max"),
        mae_pct_p75=("mae_pct", lambda x: x.quantile(0.75)),
    )
    .sort_values("mae_pct_mean", ascending=False)
)

summary["mae_pct_mean"]   = summary["mae_pct_mean"].map("{:.2f}%".format)
summary["mae_pct_median"] = summary["mae_pct_median"].map("{:.2f}%".format)
summary["mae_pct_max"]    = summary["mae_pct_max"].map("{:.2f}%".format)
summary["mae_pct_p75"]    = summary["mae_pct_p75"].map("{:.2f}%".format)

summary.columns = ["Trades", "Avg MAE %", "Median MAE %", "Max MAE %", "P75 MAE %"]
summary.to_clipboard()


In [ ]:
mfe_rows = []
# We look at all trades now, not just winners, to see missed opportunities
for _, trade in trades_df.iterrows():
    bars = bars_1m_by_symbol.get(trade["symbol"])
    if bars is None: continue
    
    entry_t = pd.to_datetime(trade["entry_time"], utc=True)
    # Define the "Look-ahead" window: Entry + 8 Hours
    end_window_t = entry_t + pd.Timedelta(hours=8)
    
    # Slice the data for the 8-hour period following entry
    window_bars = bars.loc[entry_t : end_window_t]
    
    if window_bars.empty: continue

    if trade["direction"] == "LONG":
        # Best case for a Long is the High
        best_price = window_bars["high"].max()
        mfe_price = best_price - trade["entry"]
    else:
        # Best case for a Short is the Low
        best_price = window_bars["low"].min()
        mfe_price = trade["entry"] - best_price

    mfe_pct = mfe_price / trade["entry"] * 100
    mfe_rows.append({
        "symbol": trade["symbol"], 
        "mfe_pct": mfe_pct, 
        "hour_of_day": entry_t.hour
    })

mfe_df = pd.DataFrame(mfe_rows)

summary = (
    mfe_df.groupby("symbol")
    .agg(
        trades=("mfe_pct", "size"),
        mfe_pct_mean=("mfe_pct", "mean"),
        mfe_pct_median=("mfe_pct", "median"),
        mfe_pct_max=("mfe_pct", "max"),
        mfe_pct_p75=("mfe_pct", lambda x: x.quantile(0.75)),
    )
    .sort_values("mfe_pct_mean", ascending=False)
)

summary["mfe_pct_mean"]   = summary["mfe_pct_mean"].map("{:.2f}%".format)
summary["mfe_pct_median"] = summary["mfe_pct_median"].map("{:.2f}%".format)
summary["mfe_pct_max"]    = summary["mfe_pct_max"].map("{:.2f}%".format)
summary["mfe_pct_p75"]    = summary["mfe_pct_p75"].map("{:.2f}%".format)


summary.columns = ["Trades", "Avg MFE %", "Median MFE %", "Max MFE %", "P75 MFE %"]
summary

In [ ]:
import numpy as np

vol_rows = []
for _, trade in trades_df.iterrows():
    bars = bars_1m_by_symbol.get(trade["symbol"])
    if bars is None: continue
    
    entry_t = pd.to_datetime(trade["entry_time"], utc=True)
    # Define the 8-hour lookback window (480 minutes)
    start_lookback_t = entry_t - pd.Timedelta(hours=8)
    
    # Slice the data for the 8 hours BEFORE entry
    pre_trade_bars = bars.loc[start_lookback_t : entry_t]
    
    if len(pre_trade_bars) < 60: # Ensure we have at least 1 hour of data to be meaningful
        continue

    # Calculate 1-minute log returns
    log_returns = np.log(pre_trade_bars["close"] / pre_trade_bars["close"].shift(1))
    
    # Standard deviation of returns (Volatility)
    # We multiply by sqrt(480) to scale the 1m vol to the 8h window
    volatility_8h = log_returns.std() * np.sqrt(480) * 100 # Expressed as %
    
    # Also get the 8h High-Low Range for context
    high_low_range = (pre_trade_bars["high"].max() - pre_trade_bars["low"].min()) / pre_trade_bars["low"].min() * 100

    vol_rows.append({
        "symbol": trade["symbol"],
        "entry_time": entry_t,
        "pre_entry_vol": volatility_8h,
        "pre_entry_range": high_low_range,

    })

vol_df = pd.DataFrame(vol_rows)


summary = (
    vol_df.groupby("symbol")
    .agg(
        trades=("pre_entry_vol", "size"),
        pre_entry_vol_mean=("pre_entry_vol", "mean"),
        pre_entry_vol_median=("pre_entry_vol", "median"),
        pre_entry_vol_max=("pre_entry_vol", "max"),
        pre_entry_vol_p75=("pre_entry_vol", lambda x: x.quantile(0.75)),
    )
    .sort_values("pre_entry_vol_mean", ascending=False)
)

summary["pre_entry_vol_mean"]   = summary["pre_entry_vol_mean"].map("{:.2f}%".format)
summary["pre_entry_vol_median"] = summary["pre_entry_vol_median"].map("{:.2f}%".format)
summary["pre_entry_vol_max"]    = summary["pre_entry_vol_max"].map("{:.2f}%".format)
summary["pre_entry_vol_p75"]    = summary["pre_entry_vol_p75"].map("{:.2f}%".format)


summary.columns = ["Trades", "Avg Pre-Entry Vol %", "Median Pre-Entry Vol %", "Max Pre-Entry Vol %", "P75 Pre-Entry Vol %"]
summary

In [ ]:
summary.to_clipboard()